In [1]:
import findspark
import configparser

In [2]:
# /opt/manual/spark: this is SPARK_HOME path
findspark.init("/opt/manual/spark")

In [3]:
from pyspark.sql import SparkSession, functions as F

In [4]:
! ls -l $SPARK_HOME/jars | grep postgresql

-rw-rw-r--. 1 train train   932808 Aug 14  2021 postgresql-42.2.14.jar


In [5]:
# If you want to connect a rdbms, related driver should be in the classpath

In [5]:
spark = SparkSession.builder \
.appName("JDBC and SQL") \
.master("local[4]") \
.getOrCreate()

2023-01-29 21:50:55,975 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# Read from file

In [6]:
market5mil_parquet = spark.read.load("file:///home/train/datasets/market5mil_parquet")

In [7]:
config = configparser.RawConfigParser()

config.read('./db_conn')
user_name = config.get('DB', 'user_name')
password = config.get('DB', 'password')
db_ip = config.get('DB', 'db_ip')

In [15]:
# Check postgresql
# psql -d traindb -U train -W

In [11]:
# (market5mil_parquet.write
# .format("jdbc")
# .mode("overwrite")
# .option("driver", "org.postgresql.Driver")
# .option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
# .option("dbtable", "market5mil_parquet")
# .option("user", user_name)
# .option("password", password)
# .save())

2023-01-29 18:17:32,974 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
# market5mil_parquet.limit(1).toPandas()

# Get limits

In [10]:
df_limits = (spark.read.format("jdbc")
.option("driver", "org.postgresql.Driver")
.option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
.option("query", """
select count("LOGICALREF") as count_value, min("LOGICALREF") as min_value, max("LOGICALREF") as max_value 
from market5mil_parquet
""")
.option("user", user_name)
.option("password", password)
.load())

In [11]:
df_limits.show()

+-----------+---------+---------+
|count_value|min_value|max_value|
+-----------+---------+---------+
|    5387858|        1|  5388261|
+-----------+---------+---------+



# Read From Postgresql without partitions

In [16]:
import time

start_time = time.time()

df_without_partitions = (spark.read.format("jdbc")
.option("driver", "org.postgresql.Driver")
.option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
.option("dbtable", "market5mil_parquet")
.option("user", user_name)
.option("password", password)
.load())

df_without_partitions.count()

print("----- %s secs -----" %(time.time() - start_time))

----- 9.326244592666626 secs -----


# Read with partitions

In [17]:
import time

start_time = time.time()

df_with_partitions = (spark.read.format("jdbc")
.option("driver", "org.postgresql.Driver")
.option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
.option("dbtable", "market5mil_parquet")
.option("user", user_name)
.option("partitionColumn", "LOGICALREF")
.option("numPartitions", "4")
.option("lowerBound", "1")
.option("upperBound", "5388261")
.option("password", password)
.load())

df_with_partitions.count()

print("----- %s secs -----" %(time.time() - start_time))



----- 8.969757556915283 secs -----


# Write to Postgresql with partitions

In [8]:
import time

start_time = time.time()

(market5mil_parquet.write
.format("jdbc")
.mode("overwrite")
.option("driver", "org.postgresql.Driver")
.option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
.option("dbtable", "market5mil_with_partitions")
.option("user", user_name)
.option("password", password)
.option("partitionColumn", "LOGICALREF")
.option("numPartitions", "4")
.option("lowerBound", "1")
.option("upperBound", "5388261")
.save())

print("----- %s secs -----" %(time.time() - start_time))

2023-01-29 21:51:30,905 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


----- 1020.6199526786804 secs -----


# Write Postgresql without partitions

In [9]:
import time

start_time = time.time()

(market5mil_parquet.write
.format("jdbc")
.mode("overwrite")
.option("driver", "org.postgresql.Driver")
.option("url", f"jdbc:postgresql://{db_ip}:5432/traindb")
.option("dbtable", "market5mil_parquet")
.option("user", user_name)
.option("password", password)
.save())

print("----- %s secs -----" %(time.time() - start_time))

----- 988.0005991458893 secs -----


In [27]:
spark.stop()